# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevinwdt/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason code

I confirm **Lane 4: CTR / Engagement Opportunity Scoring**.

My baseline rule identifies pages that receive meaningful search exposure but have a measured CTR below the median CTR of pages in a similar average-position bucket.

The rule works as follows:

1. Keep pages with at least 500 impressions.
2. Keep pages with an average position between 1 and 20.
3. Group pages into similar position buckets: positions 1–3, 4–10, and 11–20.
4. Calculate the median CTR in each position bucket.
5. Keep pages whose CTR is below their position-bucket median.
6. Rank them by the estimated size of the missed-click opportunity.

The score is:

**baseline score = (positive CTR gap ÷ 100) × impressions**

The rule produces one reason code:

`CTR_BELOW_POSITION_BENCHMARK`

The action label is:

`REVIEW_SNIPPET_AND_INTENT`

This means an SEO specialist or content editor should review the page's title, meta description, search-intent match, and content. It does not mean the page should automatically be changed.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "kevinwdt/flyrank-internship-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset loaded successfully.")
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


Dataset loaded successfully.
Rows: 30,000
Columns: 44


### Signal check 1: CTR versus average position

My rule assumes that CTR should be interpreted relative to average search position. A CTR that is weak for a page in position 2 may be normal for a page in position 18.

I will compare measured CTR across position buckets. I require at least 100 impressions to reduce low-volume noise.

In [25]:
signal_df = df[
    (df["impressions_90d"] >= 100)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 50)
].copy()

signal_df["position_bucket"] = pd.cut(
    signal_df["avg_position"],
    bins=[0, 3, 10, 20, 50],
    labels=["1-3", "4-10", "11-20", "21-50"],
    include_lowest=True
)

position_table = (
    signal_df.groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_ctr_pct=("ctr", "median"),
        mean_ctr_pct=("ctr", "mean"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

display(position_table.round(3))

ctr_values = position_table["median_ctr_pct"]

if ctr_values.is_monotonic_decreasing:
    position_verdict = "CONFIRMED"
elif ctr_values.is_monotonic_increasing:
    position_verdict = "OPPOSITE"
else:
    position_verdict = "MIXED"

print("SIGNAL 1 VERDICT:", position_verdict)

,position_bucket,n,median_ctr_pct,mean_ctr_pct,median_impressions
0,1-3,555,0.19,0.337,3047.0
1,4-10,8660,0.23,0.354,2940.0
2,11-20,5876,0.15,0.256,1377.0
3,21-50,6037,0.06,0.142,1209.0


SIGNAL 1 VERDICT: MIXED


**Signal 1 verdict: MIXED**

Measured median CTR differs across average-position buckets, but the pattern is not perfectly consistent across every bucket. This means position should still be considered when judging CTR, but I should not claim that CTR always decreases smoothly as the position number increases.

### Signal check 2: Impression volume and opportunity size

My rule assumes that impression volume should affect review priority.

A small CTR gap on a page with thousands of impressions may represent a larger opportunity than the same CTR gap on a page with only a few impressions.

The estimated missed-click value is only a prioritization measure. It does not prove that editing a page would recover those clicks.

In [26]:
# Calculate the position-adjusted CTR benchmark.
signal_df["expected_ctr_pct"] = (
    signal_df.groupby("position_bucket", observed=True)["ctr"]
    .transform("median")
)

# Keep only the amount by which CTR falls below the benchmark.
signal_df["positive_ctr_gap_pp"] = (
    signal_df["expected_ctr_pct"] - signal_df["ctr"]
).clip(lower=0)

# CTR is stored in percentage points.
# For example, 0.50 means 0.50%, so divide by 100.
signal_df["estimated_missed_clicks"] = (
    signal_df["positive_ctr_gap_pp"] / 100
    * signal_df["impressions_90d"]
)

signal_df["volume_bucket"] = pd.cut(
    signal_df["impressions_90d"],
    bins=[99, 499, 1999, np.inf],
    labels=["100-499", "500-1,999", "2,000+"],
    include_lowest=True
)

# Test volume only among pages that are below their position benchmark,
# because these are the pages eligible for the baseline queue.
volume_candidates = signal_df[
    signal_df["positive_ctr_gap_pp"] > 0
].copy()

volume_candidates["volume_bucket"] = pd.cut(
    volume_candidates["impressions_90d"],
    bins=[99, 499, 1999, np.inf],
    labels=["100-499", "500-1,999", "2,000+"],
    include_lowest=True
)

volume_table = (
    volume_candidates.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr_gap_pp=("positive_ctr_gap_pp", "median"),
        median_estimated_missed_clicks=(
            "estimated_missed_clicks",
            "median"
        )
    )
    .reset_index()
)

display(volume_table.round(3))

opportunity_values = volume_table[
    "median_estimated_missed_clicks"
]

if opportunity_values.is_monotonic_increasing:
    volume_verdict = "CONFIRMED"
elif opportunity_values.is_monotonic_decreasing:
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

print("SIGNAL 2 VERDICT:", volume_verdict)

,volume_bucket,n,median_impressions,median_ctr_gap_pp,median_estimated_missed_clicks
0,100-499,3149,237.0,0.15,0.264
1,"500-1,999",3169,967.0,0.08,0.893
2,"2,000+",4000,5803.5,0.07,4.452


SIGNAL 2 VERDICT: CONFIRMED


**Signal 2 verdict: CONFIRMED**

Higher-impression buckets have a larger measured missed-click opportunity. This supports using impression volume in the baseline score. However, the estimated opportunity does not prove that an edit would produce those clicks.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

The baseline keeps pages with at least 500 impressions and an average position between 1 and 20.

A page receives a positive score only when its measured CTR is below the median CTR for its position bucket.

The queue is ranked by estimated missed clicks:

**baseline score = CTR gap ÷ 100 × impressions**

Every selected page receives one reason code and one action label.

This is a transparent baseline for human review. It is not a prediction that a specific edit will cause additional clicks.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Keep only pages that have enough exposure and useful positions.
rule_df = signal_df[
    (signal_df["impressions_90d"] >= 500)
    & (signal_df["avg_position"] >= 1)
    & (signal_df["avg_position"] <= 20)
].copy()

rule_df["baseline_score"] = (
    rule_df["positive_ctr_gap_pp"] / 100
    * rule_df["impressions_90d"]
)

# Keep only pages performing below their position benchmark.
queue = rule_df[
    rule_df["baseline_score"] > 0
].copy()

queue["reason_code"] = "CTR_BELOW_POSITION_BENCHMARK"
queue["action_label"] = "REVIEW_SNIPPET_AND_INTENT"

queue = queue.sort_values(
    by=["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

queue_columns = [
    "rank",
    "content_id",
    "content_type",
    "main_intent",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "position_bucket",
    "ctr",
    "expected_ctr_pct",
    "positive_ctr_gap_pp",
    "baseline_score",
    "reason_code",
    "action_label"
]

queue = queue[queue_columns]

print("Pages in ranked queue:", f"{len(queue):,}")
display(queue.head(20).round(3))


Pages in ranked queue: 5,524


,rank,content_id,content_type,main_intent,impressions_90d,clicks_90d,avg_position,position_bucket,ctr,expected_ctr_pct,positive_ctr_gap_pp,baseline_score,reason_code,action_label
0,1,content_36ff89c8214e,keyword article,informational,295097,154,7.3,4-10,0.05,0.23,0.18,531.175,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
1,2,content_c8e9d6ab9013,keyword article,informational,208678,0,9.7,4-10,0.00,0.23,0.23,479.959,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
2,3,content_5fe46e04994d,keyword article,informational,517715,741,4.2,4-10,0.14,0.23,0.09,465.944,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
3,4,content_c84a0ab98e90,keyword article,informational,223271,70,7.8,4-10,0.03,0.23,0.20,446.542,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
4,5,content_8451fc6f034d,keyword article,informational,272144,75,2.3,1-3,0.03,0.19,0.16,435.430,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
5,6,content_453722754fea,keyword article,informational,140079,16,7.6,4-10,0.01,0.23,0.22,308.174,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
6,7,content_73c54f78c06a,keyword article,informational,213963,211,4.7,4-10,0.10,0.23,0.13,278.152,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
7,8,content_91652435f57a,keyword article,commercial,159590,100,7.8,4-10,0.06,0.23,0.17,271.303,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
8,9,content_c1fe78bc4e37,keyword article,commercial,134055,43,7.5,4-10,0.03,0.23,0.20,268.110,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT
9,10,content_0919dd345d80,keyword article,informational,119217,26,7.0,4-10,0.02,0.23,0.21,250.356,CTR_BELOW_POSITION_BENCHMARK,REVIEW_SNIPPET_AND_INTENT


In [28]:
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "baseline_action_score.csv"

queue.to_csv(csv_path, index=False)

print("CSV created successfully:")
print(csv_path)
print("Rows written:", f"{len(queue):,}")

CSV created successfully:
work/outputs/baseline_action_score.csv
Rows written: 5,524


In [29]:
metrics = {
    "lane": "CTR / Engagement Opportunity Scoring",
    "signal_1_verdict": position_verdict,
    "signal_2_verdict": volume_verdict,
    "minimum_impressions": 500,
    "position_range": "1-20",
    "queue_rows": int(len(queue)),
    "reason_code": "CTR_BELOW_POSITION_BENCHMARK",
    "action_label": "REVIEW_SNIPPET_AND_INTENT"
}

metrics_path = output_dir / "baseline_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print("Metrics receipt written to:")
print(metrics_path)

Metrics receipt written to:
work/outputs/baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

I reviewed the first 20 recommendations produced by the baseline.

For every page, I considered:

- the recommended action;
- the reason it appeared in the queue;
- how confident the rule appears;
- what information could make the recommendation wrong.

The score shows review priority, not proof that the page should be edited.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

def confidence_note(row):
    if row["clicks_90d"] == 0:
        return (
            "Medium confidence: the exposure and CTR gap are large, "
            "but zero clicks could reflect query mismatch or a data-quality issue."
        )

    if (
        row["impressions_90d"] >= 2000
        and row["positive_ctr_gap_pp"] >= 0.20
    ):
        return "Higher confidence: strong exposure and a large CTR gap."

    if row["impressions_90d"] >= 1000:
        return (
            "Medium confidence: meaningful exposure, "
            "but the cause of the CTR gap is uncertain."
        )

    return (
        "Lower confidence: enough exposure to review, "
        "but the measured opportunity is smaller."
    )

def wrong_reason(row):
    if row["avg_position"] > 10:
        return (
            "Its CTR may be normal for lower search positions, "
            "query intent, or SERP features."
        )

    if row["clicks_90d"] == 0:
        return (
            "The impressions may come from irrelevant queries or "
            "there may be a tracking/data-quality issue."
        )

    return (
        "Brand effects, query intent, or SERP features may explain "
        "the CTR gap rather than the title or content."
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

top20["why_it_is_here"] = top20.apply(
    lambda row: (
        f"{row['impressions_90d']:.0f} impressions, "
        f"position {row['avg_position']:.1f}, "
        f"CTR {row['ctr']:.3f}% versus "
        f"{row['expected_ctr_pct']:.3f}% benchmark."
    ),
    axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])


,rank,content_id,action_label,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_36ff89c8214e,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"295097 impressions, position 7.3, CTR 0.050% v...","Medium confidence: meaningful exposure, but th...","Brand effects, query intent, or SERP features ..."
1,2,content_c8e9d6ab9013,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"208678 impressions, position 9.7, CTR 0.000% v...",Medium confidence: the exposure and CTR gap ar...,The impressions may come from irrelevant queri...
2,3,content_5fe46e04994d,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"517715 impressions, position 4.2, CTR 0.140% v...","Medium confidence: meaningful exposure, but th...","Brand effects, query intent, or SERP features ..."
3,4,content_c84a0ab98e90,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"223271 impressions, position 7.8, CTR 0.030% v...",Higher confidence: strong exposure and a large...,"Brand effects, query intent, or SERP features ..."
4,5,content_8451fc6f034d,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"272144 impressions, position 2.3, CTR 0.030% v...","Medium confidence: meaningful exposure, but th...","Brand effects, query intent, or SERP features ..."
5,6,content_453722754fea,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"140079 impressions, position 7.6, CTR 0.010% v...",Higher confidence: strong exposure and a large...,"Brand effects, query intent, or SERP features ..."
6,7,content_73c54f78c06a,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"213963 impressions, position 4.7, CTR 0.100% v...","Medium confidence: meaningful exposure, but th...","Brand effects, query intent, or SERP features ..."
7,8,content_91652435f57a,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"159590 impressions, position 7.8, CTR 0.060% v...","Medium confidence: meaningful exposure, but th...","Brand effects, query intent, or SERP features ..."
8,9,content_c1fe78bc4e37,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"134055 impressions, position 7.5, CTR 0.030% v...",Higher confidence: strong exposure and a large...,"Brand effects, query intent, or SERP features ..."
9,10,content_0919dd345d80,REVIEW_SNIPPET_AND_INTENT,CTR_BELOW_POSITION_BENCHMARK,"119217 impressions, position 7.0, CTR 0.020% v...",Higher confidence: strong exposure and a large...,"Brand effects, query intent, or SERP features ..."


In [31]:
for _, row in top20.iterrows():
    print(
        f"{int(row['rank'])}. "
        f"Action: {row['action_label']} | "
        f"Reason: {row['reason_code']} | "
        f"Why: {row['why_it_is_here']} | "
        f"Confidence: {row['confidence_note']} | "
        f"Could be wrong if: {row['what_would_make_it_wrong']}"
    )

1. Action: REVIEW_SNIPPET_AND_INTENT | Reason: CTR_BELOW_POSITION_BENCHMARK | Why: 295097 impressions, position 7.3, CTR 0.050% versus 0.230% benchmark. | Confidence: Medium confidence: meaningful exposure, but the cause of the CTR gap is uncertain. | Could be wrong if: Brand effects, query intent, or SERP features may explain the CTR gap rather than the title or content.
2. Action: REVIEW_SNIPPET_AND_INTENT | Reason: CTR_BELOW_POSITION_BENCHMARK | Why: 208678 impressions, position 9.7, CTR 0.000% versus 0.230% benchmark. | Confidence: Medium confidence: the exposure and CTR gap are large, but zero clicks could reflect query mismatch or a data-quality issue. | Could be wrong if: The impressions may come from irrelevant queries or there may be a tracking/data-quality issue.
3. Action: REVIEW_SNIPPET_AND_INTENT | Reason: CTR_BELOW_POSITION_BENCHMARK | Why: 517715 impressions, position 4.2, CTR 0.140% versus 0.230% benchmark. | Confidence: Medium confidence: meaningful exposure, but the c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks and leakage check

The lowest-scoring pages within the top 20 are the weakest recommendations in this review set. They may have lower impression volume, a smaller CTR gap, or search positions where CTR is naturally more difficult to interpret.

The baseline uses only current measured inputs:

- impressions during the 90-day window;
- measured CTR during the same window;
- average position during the same window.

It does not use a future outcome, a label-derived feature, or a precomputed FlyRank product flag.

The rule remains directional decision support. It cannot determine why CTR is low or prove that changing a page will improve clicks.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Weakest five recommendations inside the top 20:")

weak_picks = top20.tail(5)[
    [
        "rank",
        "content_id",
        "impressions_90d",
        "avg_position",
        "ctr",
        "expected_ctr_pct",
        "positive_ctr_gap_pp",
        "baseline_score"
    ]
]

display(weak_picks.round(3))

Weakest five recommendations inside the top 20:


,rank,content_id,impressions_90d,avg_position,ctr,expected_ctr_pct,positive_ctr_gap_pp,baseline_score
15,16,content_cb112fce36be,309910,5.6,0.16,0.23,0.07,216.937
16,17,content_4c76e9b13aea,127952,7.4,0.07,0.23,0.16,204.723
17,18,content_8c19996aa890,509252,2.5,0.15,0.19,0.04,203.701
18,19,content_63f88d16fdb8,99013,6.4,0.03,0.23,0.20,198.026
19,20,content_e12868d1f396,149712,2.9,0.07,0.19,0.12,179.654


In [33]:
# Only these columns contribute to the score.
score_inputs = {
    "impressions_90d",
    "avg_position",
    "ctr"
}

forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "low_ctr_label",
    "outcome_ctr_pct",
    "outcome_ctr_gap",
    "future_ctr",
    "future_clicks",
    "quick_win_flag",
    "refresh_flag",
    "ctr_fix_flag"
}

leaking_inputs = score_inputs.intersection(forbidden_inputs)

assert not leaking_inputs, (
    f"Leakage found in score inputs: {sorted(leaking_inputs)}"
)

print("Score inputs:", sorted(score_inputs))
print("Leakage check passed.")
print("No future-window, label-derived, or product-flag inputs were used.")

Score inputs: ['avg_position', 'ctr', 'impressions_90d']
Leakage check passed.
No future-window, label-derived, or product-flag inputs were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.